# Notebook 4 — Influences (Causal Relationships)

An `Influence` expresses a **causal edge** between two columns: when the source column has a value, the target column's value changes by a specified amount. This is how Blueprint encodes domain knowledge like:

- Larger homes have higher prices
- High crime areas have lower property values
- Members receive a discount
- Revenue scales with units sold

Influences are applied **after** class overrides and **before** computed features. Blueprint resolves a topological sort of all source → target edges to determine evaluation order, so influences can safely chain (price depends on sqft, margin depends on price, etc.).

The three main knobs on an `Influence`:
1. **`effect`** — a string describing how the target changes
2. **`by_class`** — different effect magnitudes for different population segments
3. **`fn`** — a callable for arbitrary transformations

In [1]:
from blueprint import Blueprint, Feature, Class, Influence
from blueprint.presets.influences import ScalesWith, CorrelatedWith, Caps

import pandas as pd
import numpy as np

## The `Influence` pattern

```python
Influence("source").on("target", effect="...", by_class={...}, fn=..., when=...)
```

- `source` — name of the column that drives the change
- `target` — name of the column that receives the change
- At least one of `effect`, `by_class`, or `fn` is required

`.on()` returns `self`, so one `Influence` can target multiple columns by chaining:

In [2]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("sqft",         dtype=int,   base=2_000, std=400, clip=(700, None)),
    Feature("price",        dtype=float, base=300_000, std=50_000, clip=(50_000, None)),
    Feature("monthly_rent", dtype=float, base=2_000,   std=400,    clip=(500, None)),
)

# One Influence, two targets — chained with .on()
bp.add_influence(
    Influence("sqft")
    .on("price",        effect="+110 per unit")
    .on("monthly_rent", effect="+0.5 per unit")
)

df = bp.emit()
print(df.head(6).round(0))
print()
print("price vs sqft correlation:", np.corrcoef(df.sqft, df.price)[0, 1].round(3))
print("rent  vs sqft correlation:", np.corrcoef(df.sqft, df.monthly_rent)[0, 1].round(3))

   sqft     price  monthly_rent
0  2122  601613.0        3037.0
1  1584  518999.0        2500.0
2  2300  517026.0        2984.0
3  2376  486235.0        3442.0
4  1220  285974.0        2611.0
5  1479  435515.0        2876.0

price vs sqft correlation: 0.634
rent  vs sqft correlation: 0.463


## Effect strings

The `effect` parameter is a compact string that Blueprint parses into one of six operations. All six are listed below with the exact syntax and its mathematical meaning.

| Syntax | Operation | Formula |
|---|---|---|
| `"+5000"` | Flat additive | `target += 5000` |
| `"+12%"` | Percentage | `target *= 1.12` |
| `"+110 per unit"` | Per-unit additive | `target += 110 × source` |
| `"-0.6% per unit"` | Per-unit percentage | `target *= (1 − 0.006 × source)` |
| `"=75000"` | Set (overwrite) | `target = 75000` |
| `"*1.5"` | Multiply | `target *= 1.5` |

Signs work as expected: `"-5000"` subtracts, `"-12%"` reduces, `"+0.6% per unit"` grows with the source.

### Flat additive: `"+N"`

Adds a fixed constant to every affected row. When the source is a **boolean** column, the effect is automatically restricted to `True` rows (see the Boolean sources section for details).

In [3]:
bp = Blueprint(n=8, seed=42)
bp.add_feature(
    Feature("has_garage", dtype=bool,  p=0.6),
    Feature("price",      dtype=float, base=250_000, std=0),  # std=0 isolates the effect
)
bp.add_influence(Influence("has_garage").on("price", effect="+8000"))

df = bp.emit()
print(df)
print()
print("no-garage mean:", df[~df.has_garage]["price"].mean())
print("garage mean:   ", df[df.has_garage]["price"].mean())

   has_garage     price
0       False  250000.0
1        True  258000.0
2       False  250000.0
3       False  250000.0
4        True  258000.0
5       False  250000.0
6       False  250000.0
7       False  250000.0

no-garage mean: 250000.0
garage mean:    258000.0


### Percentage: `"+N%"` or `"-N%"`

Multiplies the target by `(1 + N/100)`. The source column determines which rows are affected — if boolean, only `True` rows; if numeric (without a `when=` gate), all rows.

In [4]:
bp = Blueprint(n=8, seed=42)
bp.add_feature(
    Feature("has_pool", dtype=bool,  p=0.4),
    Feature("price",    dtype=float, base=300_000, std=0),
)
bp.add_influence(Influence("has_pool").on("price", effect="+12%"))

df = bp.emit()
print(df)
print()
print("no-pool mean:", df[~df.has_pool]["price"].mean())
delta = df[df.has_pool]["price"].mean() / df[~df.has_pool]["price"].mean() - 1
print(f"pool mean:    {df[df.has_pool]['price'].mean():.1f}   ({delta:+.1%})")

   has_pool     price
0     False  300000.0
1     False  300000.0
2     False  300000.0
3     False  300000.0
4      True  336000.0
5     False  300000.0
6     False  300000.0
7     False  300000.0

no-pool mean: 300000.0
pool mean:    336000.0   (+12.0%)


### Per-unit additive: `"+N per unit"`

The source column's value directly scales the magnitude: `target += N × source`. This is the natural choice when the target is proportional to a quantity (revenue ∝ units, price ∝ sqft).

Pair this with `derived=True` on the target when the column has no baseline of its own — it accumulates only from influence effects.

In [5]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature("sqft",    dtype=int,   base=2_000, std=400, clip=(700, None)),
    Feature("revenue", dtype=float, base=0,     std=0,   derived=True),
)
bp.add_influence(Influence("sqft").on("revenue", effect="+110 per unit"))

df = bp.emit()
print(df)
print()
print("revenue / sqft (always 110.0):", (df.revenue / df.sqft).round(1).to_list())

   sqft   revenue
0  2122  233420.0
1  1584  174240.0
2  2300  253000.0
3  2376  261360.0
4  1220  134200.0
5  1479  162690.0

revenue / sqft (always 110.0): [110.0, 110.0, 110.0, 110.0, 110.0, 110.0]


### Per-unit percentage: `"-N% per unit"`

Each unit of the source column changes the target by `N%`: `target *= (1 + N/100 × source)`. Use this when the penalty or bonus *compounds* with the source's magnitude — crime rate depressing price, distance increasing cost, etc.

In [6]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature("crime_rate", dtype=float, base=20, std=8, clip=(0, 80)),
    Feature("price",      dtype=float, base=300_000, std=0),
)
bp.add_influence(Influence("crime_rate").on("price", effect="-0.6% per unit"))

df = bp.emit()
print(df.round(0))
print()
print("Formula: price = 300,000 \u00d7 (1 \u2212 0.006 \u00d7 crime_rate)")
expected = 300_000 * (1 - 0.006 * df["crime_rate"])
print("Matches exact formula:", np.allclose(df["price"], expected))

   crime_rate     price
0        22.0  259612.0
1        12.0  278976.0
2        26.0  253194.0
3        28.0  250456.0
4         4.0  292095.0
5        10.0  282751.0

Formula: price = 300,000 × (1 − 0.006 × crime_rate)
Matches exact formula: True


### Set: `"=N"`

Overwrites the target to a fixed value on affected rows. Useful for flag columns or when a condition should completely replace the existing value.

In [7]:
bp = Blueprint(n=8, seed=42)
bp.add_feature(
    Feature("is_blocked", dtype=bool,  p=0.3),
    Feature("status",     dtype=float, base=1, std=0),
)
bp.add_influence(Influence("is_blocked").on("status", effect="=0"))

df = bp.emit()
print(df)
print()
print("Non-blocked status always 1:", (df[~df.is_blocked]["status"] == 1.0).all())
print("Blocked status always 0:    ", (df[df.is_blocked]["status"] == 0.0).all())

   is_blocked  status
0       False     1.0
1       False     1.0
2       False     1.0
3       False     1.0
4        True     0.0
5       False     1.0
6       False     1.0
7       False     1.0

Non-blocked status always 1: True
Blocked status always 0:     True


### Multiply: `"*N"`

Multiplies the target by a constant factor. Use when the target should scale to a fixed multiple for affected rows — a surcharge tier, a fraud multiplier, a premium marker.

In [8]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature("is_premium", dtype=bool,  p=0.4),
    Feature("base_price", dtype=float, base=100_000, std=20_000),
    Feature("price",      dtype=float, base=100_000, std=20_000),
)
bp.add_influence(Influence("is_premium").on("price", effect="*1.3"))

df = bp.emit()
print(df[["is_premium", "price"]].round(0))
print()
# base_price and price start with identical values (same seed, same params)
ratio_non = (df[~df.is_premium]["price"] / df[~df.is_premium]["base_price"]).mean()
ratio_prem = (df[df.is_premium]["price"] / df[df.is_premium]["base_price"]).mean()
print(f"Non-premium / base price ratio (should ~1.0): {ratio_non:.1f}")
print(f"Premium    / base price ratio (should ~1.3): {ratio_prem:.1f}")

   is_premium     price
0       False  102557.0
1       False   93675.0
2       False   99664.0
3       False   82939.0
4        True  152864.0
5       False  115556.0

Non-premium / base price ratio (should ~1.0): 1.0
Premium    / base price ratio (should ~1.3): 1.3


## Boolean sources

When the source column is `dtype=bool`, Blueprint automatically restricts the influence to rows where the source is `True`. You don’t need a `when=` gate — the boolean column itself is the gate.

This is the standard pattern for “feature flag” columns: `has_pool`, `is_member`, `has_garage`, `is_flagged`.

In [9]:
bp = Blueprint(n=8, seed=42)
bp.add_feature(
    Feature("is_member", dtype=bool,  p=0.5),
    Feature("price",     dtype=float, base=100, std=0),
)
# Boolean source: -15% fires only on rows where is_member == True
bp.add_influence(Influence("is_member").on("price", effect="-15%"))

df = bp.emit()
print(df)
print()
print("Non-member price always 100:", (df[~df.is_member]["price"] == 100.0).all())
print("Member price always 85:     ", (df[df.is_member]["price"] == 85.0).all())

   is_member  price
0      False  100.0
1       True   85.0
2      False  100.0
3      False  100.0
4       True   85.0
5      False  100.0
6      False  100.0
7      False  100.0

Non-member price always 100: True
Member price always 85:      True


## Class-conditional effects with `by_class`

`by_class` lets the same influence apply different effects depending on which `Class` a row belongs to. The keys are class names (as registered on the Blueprint); the values are effect strings.

You can also supply a top-level `effect=` as a **default fallback** for rows that don’t match any named class in the dict.

In [10]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("neighborhood_tier", dtype="category",
            values=["luxury", "mid", "entry"], weights=[0.2, 0.5, 0.3]),
    Feature("has_pool", dtype=bool,  p=0.3),
    Feature("price",    dtype=float, base=300_000, std=60_000, clip=(50_000, None)),
)
bp.add_class(
    Class("luxury", when=("neighborhood_tier", "==", "luxury"))
        .override("price", base=900_000, std=200_000, clip=(400_000, None)),
    Class("mid",    when=("neighborhood_tier", "==", "mid")),    # no override — label only
    Class("entry",  when=("neighborhood_tier", "==", "entry"))
        .override("price", base=160_000, std=30_000),
)

# Pool adds value in luxury/mid, but signals maintenance burden in entry
bp.add_influence(
    Influence("has_pool").on(
        "price",
        by_class={
            "luxury": "+12%",
            "mid":    "+4%",
            "entry":  "-3%",
        },
    )
)

df = bp.emit()
print("Pool premium by neighborhood tier:")
for tier in ["luxury", "mid", "entry"]:
    grp = df[df.neighborhood_tier == tier]
    w  = grp[grp.has_pool]["price"].mean()
    wo = grp[~grp.has_pool]["price"].mean()
    if not (pd.isna(w) or pd.isna(wo)):
        pct = (w / wo - 1) * 100
        print(f"  {tier:<8}pool={w:>10,.0f}  no_pool={wo:>10,.0f}  delta={pct:+.1f}%")

Pool premium by neighborhood tier:
  luxury: pool=974,070  no_pool=877,159  delta=+11.0%
  mid:    pool=321,655  no_pool=299,775  delta=+7.3%
  entry:  pool=151,766  no_pool=161,186  delta=-5.8%


### `by_class` with a default fallback

Supplying both `by_class` and `effect` handles the “everything else” case. Rows not matched by any key in `by_class` get the top-level `effect`.

In [11]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("tier",   dtype="category", values=["gold", "silver", "bronze"], weights=[1, 3, 6]),
    Feature("upsell", dtype=bool,  p=0.4),
    Feature("price",  dtype=float, base=100, std=20, clip=(0, None)),
)
bp.add_class(
    Class("gold",   when=("tier", "==", "gold")),
    Class("silver", when=("tier", "==", "silver")),
    # bronze has no registered class — handled by the default effect=
)
bp.add_influence(
    Influence("upsell").on(
        "price",
        by_class={"gold": "+20%", "silver": "+10%"},
        effect="+3%",   # fallback for bronze and any unregistered class
    )
)

df = bp.emit()
print("Upsell premium by tier:")
for tier in ["gold", "silver", "bronze"]:
    grp = df[df.tier == tier]
    w  = grp[grp.upsell]["price"].mean()
    wo = grp[~grp.upsell]["price"].mean()
    if not (pd.isna(w) or pd.isna(wo)):
        pct = (w / wo - 1) * 100
        note = "  \u2190 default effect='+3%'" if tier == "bronze" else ""
        print(f"  {tier:<8}upsell={w:.1f}  no_upsell={wo:.1f}  delta={pct:+.1f}%{note}")

Upsell premium by tier:
  gold:   upsell=115.6  no_upsell=99.5   delta=+16.2%
  silver: upsell=111.1  no_upsell=96.5   delta=+15.2%
  bronze: upsell=106.0  no_upsell=101.0  delta=+5.0%   ← default effect='+3%'


## Gated influences with `when=`

The `when=` parameter accepts the same condition forms as `Class` (tuple or callable). The influence only fires on rows satisfying the condition, independent of what the source column’s value is.

This is useful when an effect should only apply above or below a threshold — e.g., only penalise crime rates above a safe baseline.

In [12]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("crime_rate", dtype=float, base=15, std=12, clip=(0, 80)),
    Feature("price",      dtype=float, base=300_000, std=50_000, clip=(50_000, None)),
)

# The per-unit-pct penalty only fires when crime_rate > 5
bp.add_influence(
    Influence("crime_rate").on(
        "price",
        effect="-0.6% per unit",
        when=("crime_rate", ">", 5),
    )
)

df = bp.emit()
low  = df[df.crime_rate <= 5]
high = df[df.crime_rate > 5]
print("Rows with crime_rate <= 5 (gate closed, no penalty):", len(low))
print("Rows with crime_rate >  5 (gate open, penalty fires):", len(high))
print()
print("Low-crime  mean price:", low["price"].mean().round(0))
print("High-crime mean price:", high["price"].mean().round(0))

Rows with crime_rate <= 5 (gate closed, no penalty): 106
Rows with crime_rate >  5 (gate open, penalty fires): 394

Low-crime  mean price: 295283.0
High-crime mean price: 264326.0


The `when=` condition is evaluated against the live DataFrame at the time the influence is applied, so it can reference any column — including columns that were modified by earlier influences in the topological order.

`when=` also accepts a callable:

```python
Influence("income").on(
    "tax",
    effect="+35%",
    when=lambda df: df["income"] >= df["income"].quantile(0.9),
)
```

## Custom influence functions with `fn=`

`fn=` accepts a callable with the signature `fn(source_col, target_col, df) -> array`. It receives NumPy arrays and must return a new array of the same length. Use `fn=` when the standard effect strings aren’t expressive enough — non-linear curves, conditional branches, interactions between multiple columns.

In [13]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature("years_exp",   dtype=int,   base=7, std=4, clip=(0, 30)),
    Feature("base_salary", dtype=float, base=70_000, std=15_000, clip=(30_000, None)),
    Feature("salary",      dtype=float, base=70_000, std=15_000, clip=(30_000, None)),
)

def experience_bonus(source_col, target_col, df):
    # Piecewise: 2k/yr for first 5 yrs, 3k/yr for yrs 6-10, 4k/yr beyond
    bonus = np.where(
        source_col <= 5,
        source_col * 2_000,
        np.where(
            source_col <= 10,
            5 * 2_000 + (source_col - 5) * 3_000,
            5 * 2_000 + 5 * 3_000 + (source_col - 10) * 4_000
        )
    )
    return target_col + bonus

bp.add_influence(Influence("years_exp").on("salary", fn=experience_bonus))

df = bp.emit()
print("Experience bonus (non-linear fn=):")
for label, mask in [
    ("0\u20135 yrs",  df.years_exp <= 5),
    ("6\u201310 yrs", (df.years_exp > 5) & (df.years_exp <= 10)),
    ("11+ yrs",  df.years_exp > 10),
]:
    mean_bonus = (df[mask].salary - df[mask].base_salary).mean()
    print(f"  {label:<10} mean bonus = {mean_bonus:>7,.0f}")

Experience bonus (non-linear fn=):
  0–5 yrs:  mean bonus =  4,564
  6–10 yrs: mean bonus = 14,941
  11+ yrs:  mean bonus = 32,539


The `df` argument is the full DataFrame at the time the influence fires, so the function can read any other column to implement cross-column logic:

```python
def luxury_garage_bonus(source_col, target_col, df):
    # Extra $20k only for luxury homes with a garage
    luxury_mask = df["neighborhood_tier"] == "luxury"
    bonus = np.where(source_col & luxury_mask, 20_000, 0)
    return target_col + bonus

Influence("has_garage").on("price", fn=luxury_garage_bonus)
```

## Preset influences

Three preset factories cover common causal patterns. Like class presets, they return ordinary `Influence` objects — they’re thin wrappers around the same API.

```python
from blueprint.presets.influences import ScalesWith, CorrelatedWith, Caps
```

### `ScalesWith(source, target, rate)`

Shorthand for `Influence(source).on(target, effect=f"+{rate} per unit")`. The target grows (or shrinks) linearly with the source.

In [14]:
bp = Blueprint(n=1000, seed=42)
bp.add_feature(
    Feature("units_sold", dtype=int,   base=100, std=30, clip=(1, None)),
    Feature("revenue",    dtype=float, base=0,   std=0,  derived=True),
)
bp.add_influence(ScalesWith("units_sold", "revenue", rate=49.99))

df = bp.emit()
print(df.head(6))
print()
print("revenue / units always 49.99:", (df.revenue / df.units_sold).round(2).unique())

   units_sold  revenue
0         109  5448.91
1          69  3449.31
2         123  6148.77
3         128  6398.72
4          41  2049.59
5          61  3049.39

revenue / units always 49.99: [49.99]


### `CorrelatedWith(source, target, correlation)`

Imposes an **approximate** Pearson correlation between source and target by adding a normalised component of the source to the target. The actual correlation is approximately `r / √(1 + r²)` rather than exactly `r`, because the adjustment increases the target’s variance.

In [15]:
bp = Blueprint(n=2000, seed=42)
bp.add_feature(
    Feature("income",       dtype=float, base=60_000, std=20_000, clip=(15_000, None)),
    Feature("credit_score", dtype=int,   base=680,    std=80,     clip=(300, 850)),
)
bp.add_influence(CorrelatedWith("income", "credit_score", correlation=0.6))

df = bp.emit()
r = np.corrcoef(df["income"], df["credit_score"])[0, 1]
print("Target correlation parameter:", 0.6)
print("Actual Pearson r:             ", round(r, 2))
print()
print("Approximate expected r: ", round(0.6 / np.sqrt(1 + 0.36), 3), " (r / sqrt(1 + r\u00b2))")

Target correlation parameter: 0.6
Actual Pearson r:              0.52

Approximate expected r:  0.514  (r / sqrt(1 + r²))


### `Caps(source, target, threshold, decay)`

Applies a soft cap: values of the source below `threshold` leave the target unchanged; values above `threshold` progressively multiply the target by `1 / (1 + decay × excess)`, where `excess = max(source − threshold, 0)`. This models diminishing returns — distance, risk scores, saturation effects.

In [16]:
bp = Blueprint(n=1000, seed=42)
bp.add_feature(
    Feature("distance_km", dtype=float, base=10, std=8, clip=(0, 50)),
    Feature("price",       dtype=float, base=500_000, std=0),
)
# Within 5 km: no cap. Beyond 5 km: each extra km decays price by decay factor
bp.add_influence(Caps("distance_km", "price", threshold=5.0, decay=0.05))

df = bp.emit()
cuts = pd.cut(df["distance_km"], [0, 5, 10, 20, 50], include_lowest=True)
print("Mean price by distance from city centre:")
print(df.groupby(cuts, observed=True)["price"].mean().round(0))
print()
print("Within threshold (\u22645 km): price unaffected")
print("Beyond threshold:         price decays with distance")

Mean price by distance from city centre:
distance_km
(-0.001, 5.0]    500000.0
(5.0, 10.0]      443217.0
(10.0, 20.0]     346929.0
(20.0, 50.0]     259827.0
Name: price, dtype: float64

Within threshold (≤5 km): price unaffected
Beyond threshold:         price decays with distance


## Influence variability: `noise_std`

By default, influence effects are **deterministic** — every row with the same source value gets the exact same delta applied. `noise_std` makes the effect magnitude vary per row, modelling the realistic idea that the same causal factor doesn't affect every observation identically.

```python
Influence("source").on("target", effect="+155 per unit", noise_std=0.1)
```

**Semantics:** `noise_std` is the fractional standard deviation of the effect parameter. For each row, the effective rate is drawn from `N(nominal_rate, nominal_rate × noise_std)`. A `noise_std` of `0.1` means the actual rate used for that row is ±10% of the nominal value.

**Supported effect types:** `flat`, `pct`, `per_unit`, `per_unit_pct`, `multiply`. The `set` and `set_source` effects are unaffected (overwriting to a value doesn't have a meaningful magnitude to perturb).

**`fn=` edges:** when `noise_std` is set on a `fn=` edge, a seeded RNG is created and passed to the custom function as `rng=` (if the function's signature accepts it).

In [27]:
bp_det = Blueprint(n=8, seed=42)
bp_det.add_feature(
    Feature("sqft",  dtype=int,   base=1800, std=400,   clip=(500, 5000)),
    Feature("price", dtype=float, base=0,    std=0,     derived=True),
)
bp_det.add_influence(Influence("sqft").on("price", effect="+155 per unit"))

bp_noisy = Blueprint(n=8, seed=42)
bp_noisy.add_feature(
    Feature("sqft",  dtype=int,   base=1800, std=400,   clip=(500, 5000)),
    Feature("price", dtype=float, base=0,    std=0,     derived=True),
)
bp_noisy.add_influence(Influence("sqft").on("price", effect="+155 per unit", noise_std=0.1))

df_det   = bp_det.emit()
df_noisy = bp_noisy.emit()

comparison = pd.DataFrame({
    "sqft":        df_det["sqft"],
    "price_det":   df_det["price"].round(0),
    "price_noisy": df_noisy["price"].round(0),
    "effective_rate": (df_noisy["price"] / df_det["sqft"]).round(2),
})
print(comparison.to_string(index=False))
print()
print("Deterministic price std:", df_det["price"].std().round(2))
print("Noisy price std:        ", df_noisy["price"].std().round(2))

 sqft  price_det  price_noisy  effective_rate
 1922   297910.0     329770.0          171.58
 1384   214520.0     220943.0          159.64
 2100   325500.0     319046.0          151.93
 2176   337280.0     271538.0          124.79
 1020   158100.0     149041.0          146.12
 1279   198245.0     172641.0          134.98
 1851   286905.0     260891.0          140.95
 1674   259470.0     263345.0          157.31

Deterministic price std: 64018.93
Noisy price std:         64245.38


The `price_noisy` column varies around the deterministic values — the `effective_rate` column shows the per-row rate actually used (nominal is 155). The two blueprints use the same `seed=42`, so `sqft` values are identical; the only difference is the per-row rate perturbation.

Notice that the noisy price std is only slightly higher than the deterministic one: `noise_std` adds a second source of variance on top of the sqft variation, but with `noise_std=0.1` it's modest.

### Reproducibility

Noise is fully reproducible. The same `Blueprint.seed` always produces the same per-row rates. Each influence edge gets its own deterministic sub-seed derived from the master seed, the source name, and the target name — independent of every other edge and class override.

In [28]:
make = lambda seed: (
    Blueprint(n=5, seed=seed)
    .add_feature(
        Feature("x", dtype=float, base=10, std=0),
        Feature("y", dtype=float, base=0,  std=0, derived=True),
    )
    .add_influence(Influence("x").on("y", effect="+2 per unit", noise_std=0.2))
)

df1 = make(7).emit()
df2 = make(7).emit()
print("Same blueprint, same seed \u2192 identical results:", df1["y"].equals(df2["y"]))
print(df1[["x","y"]].round(3))

Same blueprint, same seed → identical results: True
      x       y
0  10.0  16.078
1  10.0  28.095
2  10.0  13.926
3  10.0  23.811
4  10.0  18.607


### `describe()` shows variability

Influences with `noise_std` are annotated in `describe()` output, making the configuration visible at a glance:

In [29]:
bp = Blueprint(n=100, seed=42)
bp.add_feature(
    Feature("sqft",  dtype=int,   base=1800, std=400, clip=(500, 5000)),
    Feature("price", dtype=float, base=0,    std=0,   derived=True),
)
bp.add_influence(Influence("sqft").on("price", effect="+155 per unit", noise_std=0.1))
bp.describe()

Blueprint(n=100, seed=42)

Features (2):
  sqft                 <class 'int'> base=1800, std=400, clip=(500, 5000)
  price                <class 'float'> base=0, std=0

Influences (1):
  sqft → price  +155 per unit  [noise_std=0.1]

Evaluation order: ['sqft', 'price']


### `fn=` edges with `noise_std`

For custom influence functions, `noise_std` creates a seeded RNG and passes it as `rng=` if the function's signature accepts it. The function is then free to use the RNG however it wants — for per-row noise, sampling from a custom distribution, or anything else. Existing 3-argument functions are called unchanged.

In [30]:
def noisy_penalty(src, tgt, df, rng=None):
    noise = rng.normal(1.0, 0.15, len(src)) if rng is not None else np.ones(len(src))
    return tgt - src * 300 * noise

bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature("distance_km", dtype=float, base=8,      std=3,     clip=(0, None)),
    Feature("price",       dtype=float, base=500000, std=80000),
)
bp.add_influence(Influence("distance_km").on("price", fn=noisy_penalty, noise_std=0.15))
print(bp.emit().round(0))

   distance_km     price
0          9.0  507597.0
1          5.0  473223.0
2         10.0  496201.0
3         11.0  427877.0
4          2.0  569800.0
5          4.0  560909.0


## Summary

| `effect` string | Operation | When to use |
|---|---|---|
| `"+N"` / `"-N"` | Flat add/subtract | Fixed premium or penalty |
| `"+N%"` / `"-N%"` | Scale by fraction | Proportional modifier |
| `"+N per unit"` | Linear scale | Target proportional to source |
| `"-N% per unit"` | Compounding decay | Sensitivity per unit of source |
| `"=N"` | Overwrite | Hard override for flagged rows |
| `"*N"` | Multiply | Fixed ratio for a class or flag |

**Key rules:**
- **Boolean source** — the effect auto-gates to `True` rows; no `when=` needed
- **`by_class`** — different effect per population segment; add `effect=` for a fallback
- **`when=`** — gates the entire influence to rows satisfying a condition
- **`fn=`** — full control; receives `(source_col, target_col, df)`, returns new target array
- Multiple `.on()` calls on one `Influence` fan the source to multiple targets

**What’s next:**
- **Notebook 5** — The Dependency DAG: how Blueprint resolves topological order, detects cycles, and exposes the evaluation graph